# VISU Predict on Colab

Train and test **V19**, the VISU traffic forecasting model, on METR-LA or PEMS-BAY under the
standard benchmark protocol. Results compare directly with published tables
([results](https://github.com/almo-intellect/visu-predict/blob/main/docs/results.md)).

**Before you start:** *Runtime → Change runtime type → GPU*. The free T4 works; an A100 or L4 is several times faster.

| Step | Time |
|---|---|
| 1-4: install, data, baselines | about 3 minutes |
| 5: train one model | 1 to 3 hours on an A100 / L4, longer on a T4 |
| 7: three seeds + ensemble | several hours (resumes after a disconnect) |

In [ ]:
# 1. Install VISU Predict
!pip install -q "visu-predict @ git+https://github.com/almo-intellect/visu-predict"
import torch
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    PRECISION = "bf16" if major >= 8 else "fp16"     # bf16 needs an Ampere GPU or newer (A100, L4)
    print("GPU:", torch.cuda.get_device_name(0), "| precision:", PRECISION)
else:
    PRECISION = "fp32"
    print("No GPU - choose Runtime > Change runtime type > GPU")


In [ ]:
# 2. Where to keep the runs. On Google Drive they survive a disconnect, and re-running
#    the training cells continues where they stopped.
USE_DRIVE = False   # set to True to keep runs on Google Drive
RUNS = "/content/runs"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/visu-predict/runs"
import os
os.makedirs(RUNS, exist_ok=True)
print("runs ->", RUNS)

In [ ]:
# 3. Download METR-LA and PEMS-BAY (160 MB)
!visu-predict download --dest /content/data

In [ ]:
# 4. Sanity check: naive baselines. On METR-LA, repeating the last reading scores 5.14 average MAE.
DATASET = "METR-LA"   # or "PEMS-BAY"
!visu-predict baselines --dataset {DATASET} --data /content/data --out "{RUNS}"

## Train one model

`--resume` continues an interrupted run with the same name, so after a disconnect just
run steps 1 to 3 again and then this cell. Add `--history-lags 288 2016` for the
day/week history variant (the best setting on PEMS-BAY).

In [ ]:
# 5. Train V19 (mixed precision + torch.compile for speed; test metrics are computed in full precision)
SEED = 42
RUN = f"{DATASET}_seed{SEED}"
!visu-predict train --dataset {DATASET} --seed {SEED} --run-name {RUN} --data /content/data --out "{RUNS}" --precision {PRECISION} --compile --save-predictions --resume

In [ ]:
# 6. Test metrics of the run
import json
res = json.load(open(f"{RUNS}/{RUN}/results.json"))
for h, name in (("h3", "15 min"), ("h6", "30 min"), ("h12", "60 min"), ("all", "average")):
    m = res["test"][h]
    print(f"{name:>8}: MAE {m['mae']:.3f}   RMSE {m['rmse']:.3f}   MAPE {m['mape']:.2f}%")
print(f"best epoch {res['best_epoch']} of {res['epochs_run']}, {res['params']:,} parameters")

## Three seeds and an ensemble

A single run can be off by 0.02-0.03 mph MAE from the next one, so the published numbers are
means over three seeds. The queue runs them side by side on the GPU, skips finished runs
(the seed-42 run above is reused) and resumes interrupted ones when the cell is run again.

In [ ]:
# 7. Seeds 42, 43, 44 + their ensemble
import json
queue = [{"name": f"{DATASET}_seed{s}", "args": f"--dataset {DATASET} --seed {s} --precision {PRECISION} --compile"}
         for s in (42, 43, 44)]
json.dump(queue, open("/content/queue.json", "w"), indent=1)
!visu-predict queue --queue /content/queue.json --data /content/data --out "{RUNS}" --max-concurrent 3 --poll 60
!visu-predict aggregate "{RUNS}"
!visu-predict ensemble "{RUNS}/{DATASET}_seed42" "{RUNS}/{DATASET}_seed43" "{RUNS}/{DATASET}_seed44"

## Everything behind the published tables

The 17 runs of [docs/results.md](https://github.com/almo-intellect/visu-predict/blob/main/docs/results.md)
(both datasets, 3 seeds, history lags, road-graph prior, weather, V18) take a day or more
of GPU time. The weather runs need `visu-predict weather --data /content/data` first.

```
!curl -sLO https://raw.githubusercontent.com/almo-intellect/visu-predict/main/configs/paper_runs.json
!visu-predict queue --queue paper_runs.json --data /content/data --out "{RUNS}" --max-concurrent 3
```